In [1]:
import os
os.chdir('application')

In [2]:
from ingest.pdf_with_tables import extract_pdf_content

In [10]:
from langchain_experimental.text_splitter import SemanticChunker

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
pdf_pages = extract_pdf_content('../assets/CP-Cuti-dan-Izin-Khusus.pdf')

In [6]:
pdf_pages[1]

{'page': 2,
 'narrative_text': '● REMOTE AREA adalah area atau lokasi kerja yang ditetapkan Manajemen sebagai\nlokasi yang jauh dari pemukiman warga atau tidak memiliki transportasi umum yang\nmemadai. 17\n18\n● SITE adalah kantor cabang dari Business Unit.\n● PERJALANAN DINAS adalah perjalanan yang dilakukan oleh karyawan ke luar lokasi\nkerja dalam rangka pelaksanaan tugas atau kepentingan Perusahaan. 19\n●\nSURAT PERINTAH PERJALANAN DINAS (SPPD) adalah surat digital atau non-digital\nyang dibuat dan disetujui Atasan sebagai bentuk pemberian tugas bagi karyawan untuk\n20\nmelakukan Perjalanan Dinas.\nCUTI TAHUNAN\n● Diberikan Kepada:\n○ 21\nKaryawan Kontrak: Diberikan 1 hari kerja setiap melewati 1 bulan masa kerja.\n○ Karyawan Tetap: Diberikan per Kuartal. Setelah bekerja 3 bulan berturut-turut,\n22\nhak cuti akan muncul sebanyak 3 hari kerja.\n● 23\nJumlah Kuota: 15 hari.\n● Jumlah Maksimal Dalam 1 Pengajuan: 12 hari. 24\n● Pengajuan dan Penggunaan Cuti Tahunan:\n○ 25\nDapat dilaku

In [5]:
# 1. Initialize the Hugging Face Embedding Model for Indonesian
def get_huggingface_embeddings():
    """Initializes and returns a Hugging Face embedding model for Indonesian."""
    model_name = "LazarusNLP/all-indo-e5-small-v4"
    model_kwargs = {'device': 'cpu'}  # Use 'cuda' for GPU
    encode_kwargs = {'normalize_embeddings': True}
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

In [8]:
# !pip install hf_xet
# !pip install -U langchain-huggingface
# !pip install -U accelerate


In [6]:
embeddings = get_huggingface_embeddings()

c:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--LazarusNLP--all-indo-e5-small-v4. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [26]:
text_splitter = SemanticChunker(
    embeddings=embeddings, 
    breakpoint_threshold_type="standard_deviation", # Other options: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=0.6 # Adjust this value to control chunk size
)

In [27]:
# The text starts by talking about Jakarta's history and then shifts to modern transportation.
jakarta_text = (
    "Jakarta, secara resmi bernama Daerah Khusus Ibukota Jakarta, adalah ibu kota negara dan kota terbesar di Indonesia. "
    "Terletak di pesisir bagian barat laut Pulau Jawa, kota ini telah menjadi pusat ekonomi, budaya, dan politik selama berabad-abad. "
    "Sejarahnya dimulai pada abad ke-4 sebagai sebuah pelabuhan perdagangan Hindu bernama Sunda Kelapa. "
    "Pada tahun 1527, Fatahillah dari Kesultanan Demak menaklukkan pelabuhan tersebut dan menamainya Jayakarta. "
    # --- The semantic chunker should identify a topic shift here ---
    "Sistem transportasi modern di Jakarta sangat bergantung pada bus dan kereta api. "
    "TransJakarta adalah sistem bus rapid transit (BRT) yang menjadi tulang punggung transportasi umum. "
    "Selain itu, MRT Jakarta dan LRT Jakarta juga telah beroperasi untuk mengurangi kemacetan lalu lintas yang parah di kota ini."
)


# 4. Create the chunks
docs = text_splitter.create_documents([jakarta_text])


# 5. View the results
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print("\n" + "="*20 + "\n")

--- Chunk 1 ---
Jakarta, secara resmi bernama Daerah Khusus Ibukota Jakarta, adalah ibu kota negara dan kota terbesar di Indonesia. Terletak di pesisir bagian barat laut Pulau Jawa, kota ini telah menjadi pusat ekonomi, budaya, dan politik selama berabad-abad.


--- Chunk 2 ---
Sejarahnya dimulai pada abad ke-4 sebagai sebuah pelabuhan perdagangan Hindu bernama Sunda Kelapa. Pada tahun 1527, Fatahillah dari Kesultanan Demak menaklukkan pelabuhan tersebut dan menamainya Jayakarta. Sistem transportasi modern di Jakarta sangat bergantung pada bus dan kereta api.


--- Chunk 3 ---
TransJakarta adalah sistem bus rapid transit (BRT) yang menjadi tulang punggung transportasi umum. Selain itu, MRT Jakarta dan LRT Jakarta juga telah beroperasi untuk mengurangi kemacetan lalu lintas yang parah di kota ini.




In [ ]:
import os
from langchain_experimental.text_splitter import SemanticChunker






# 2. Instantiate the Semantic Chunker
# The `breakpoint_threshold_type` tells the chunker how to find the topic changes.
# "percentile" is a great default. It means we'll split at the 95th percentile
# of similarity differences. In simple terms, it splits where the topic change
# is most significant.
text_splitter = SemanticChunker(
    embeddings=embeddings, 
    breakpoint_threshold_type="percentile", # Other options: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=0.95 # Adjust this value to control chunk size
)


# 3. Prepare an example text with a clear topic shift
